Logs

In [1]:
%run Utils_Log

StatementMeta(, , -1, Finished, , Finished, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
setup_log("ingesta_historico")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

Capa bronce: 
- Actua como un repositorio historico inmutable, habia la posibilidad de descargar los xml en vez de estos csv, pero influye mucho a la hora del costo porque un archivo xml peso mnucho mas teniendo etiquetas.
- Estos Archivos csv carecen de atributos dimensionales, columnas
- En esta capa no corrige esta ausencia sino en las posteriores.
- Estructura Files/Bronze/Ventas/anio=yyyy
- Idempotencia: este script esta gobernado por una lista_descarga.json es donde primero se almacenara las url si las tenemos se añaden

In [ ]:
import json
from notebookutils import mssparkutils 
from datetime import date
import time

# 1. Parámetros de la fuente
anio_inicio = 2015
anio_fin = 2025
base_path_http = "/descargas/pescafresca/pescafresca"

lista_ficheros = []

print("Iniciando escaneo físico de OneLake (Capa Bronze)...\n")
log(f"Escaneo iniciado: años {anio_inicio} a {anio_fin}")

for anio in range(anio_inicio, anio_fin + 1):
    url_origen = f"{base_path_http}{anio}.csv"
    nombre_archivo = f"pescafresca_{anio}.csv"
    
    ruta_destino = f"Files/Bronze/Ventas/anio={anio}/{nombre_archivo}"
    
    try:
        mssparkutils.fs.head(ruta_destino, 1)
        print(f"[\u2713] OMITIENDO {anio}: El archivo ya existe en los discos de OneLake.")
        log(f"OK  {anio}: archivo ya presente, se omite")
    except Exception as e:
        print(f"[!] REQUIERE DESCARGA {anio}: Añadiendo a la cola de ingesta.")
        log(f"!   {anio}: archivo ausente, se añade a la cola de descarga")
        
        lista_ficheros.append({
            "URL_Origen": url_origen,
            "Ruta_Destino_Relativa": ruta_destino,
            "Anio": str(anio)
        })

# 4. Generación y guardado del Control File
if len(lista_ficheros) > 0:
    json_datos = json.dumps(lista_ficheros, indent=4)
    ruta_json_control = "Files/Control/lista_descargas.json"
    
    # Escribimos el JSON directamente en tu carpeta de Control
    mssparkutils.fs.put(ruta_json_control, json_datos, True)
    print(f"\nArchivo de control generado en {ruta_json_control} con {len(lista_ficheros)} tareas pendientes.")
    log(f"Archivo de control generado: {len(lista_ficheros)} tareas pendientes")
else:
    print("\nEl Lakehouse está sincronizado al 100%. No se generó archivo de control.")
    mssparkutils.fs.put("Files/Control/lista_descargas.json", "[]", True)
    log("Lakehouse sincronizado al 100% (sin tareas pendientes)")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
df = spark.read.option("multiline", "true").json("Files/Control/lista_descargas.json")

display(df)
log(f"Verificación: lista_descargas.json contiene {df.count()} entradas")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### Verificacion de descargas

In [ ]:
for f in mssparkutils.fs.ls("Files/Control/logs/ingesta_historico"):
    print(f.name, "—", f.size, "bytes")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
from notebookutils import mssparkutils

#mssparkutils.session.stop()

StatementMeta(, , -1, Cancelled, , Cancelled, True)